In [11]:
import psycopg2
from pgvector.psycopg2 import register_vector
from sentence_transformers import SentenceTransformer

# !!!!! Para funcionar, você precisa ter um PostgreSQL rodando em um Docker na sua máquina local.
# Podemos pegar um container de Postgres que já vem com o pgvector instalado 
# docker run --name pgvector-demo -e POSTGRES_PASSWORD=senha_secreta -p 5432:5432 -d pgvector/pgvector:pg16

# Carregando o modelo de IA (HuggingFace)
# Esse modelo converte textos em vetores de 384 dimensões
modelo_ia = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

# Conectando ao PostgreSQL (rodando no docker)
conn = psycopg2.connect(
    dbname="postgres", 
    user="postgres", 
    password="senha_secreta", 
    host="localhost", 
    port="5432"
)
conn.autocommit = True
cur = conn.cursor()

# Habilitando o pgvector e registrando o tipo no Psycopg2
cur.execute("CREATE EXTENSION IF NOT EXISTS vector;")
register_vector(conn)

# Criando a tabela para armazenar os documentos e seus vetores
# Note que vector(384) está definido e 384 porque é o tamanho exato da saída do modelo de IA
cur.execute("""
    DROP TABLE IF EXISTS documentos;
    CREATE TABLE documentos (
        id serial PRIMARY KEY,
        conteudo text,
        embedding vector(384)
    );
""")

# Textos de exemplo
textos = [
    "O PostgreSQL é um sistema gerenciador de banco de dados relacional.",
    "Bancos NoSQL como MongoDB são baseados em documentos JSON.",
    "Para treinar uma rede neural, precisamos de GPUs de alta performance.",
    "A receita de bolo de cenoura leva cobertura de chocolate e óleo.",
    "A linguagem SQL utiliza comandos como SELECT, INSERT e JOIN."
]

# Convertendo os textos em embeddings e inserindo no banco
for texto in textos:
    # A IA transforma a frase em um array de 384 números
    vetor = modelo_ia.encode(texto).tolist() 
    
    # Inserimos o texto original e a sua representação matemática
    cur.execute(
        "INSERT INTO documentos (conteudo, embedding) VALUES (%s, %s)",
        (texto, vetor)
    )

# Testando a busca semântica 
pergunta = "Gostaria de comer uma sobremesa."
print(f"\n--- BUSCA VETORIAL ---")
print(f"Pergunta do usuário: '{pergunta}'")

# Convertemos a pergunta para vetor
vetor_pergunta = modelo_ia.encode(pergunta).tolist()

# Fazemos o SELECT usando um dos operadores mencionados (<->, <=> ou <#>)
cur.execute("""
    SELECT conteudo, embedding <-> %s::vector AS distancia 
    FROM documentos 
    ORDER BY distancia ASC;
""", (vetor_pergunta,))

resultados = cur.fetchall()

print("\nResultados encontrados (por proximidade semântica):")
for linha in resultados:
    texto_encontrado = linha[0]
    distancia = linha[1]
    print(f"-> {texto_encontrado} (Distância: {distancia:.4f})")

# Fechando a conexão
cur.close()
conn.close()


--- BUSCA VETORIAL ---
Pergunta do usuário: 'Gostaria de comer uma sobremesa.'

Resultados encontrados (por proximidade semântica):
-> A receita de bolo de cenoura leva cobertura de chocolate e óleo. (Distância: 4.7166)
-> Bancos NoSQL como MongoDB são baseados em documentos JSON. (Distância: 6.9612)
-> Para treinar uma rede neural, precisamos de GPUs de alta performance. (Distância: 7.1593)
-> A linguagem SQL utiliza comandos como SELECT, INSERT e JOIN. (Distância: 7.1714)
-> O PostgreSQL é um sistema gerenciador de banco de dados relacional. (Distância: 7.3338)
